# 신경과 재진 외래 AI 문진 — LLM 프롬프트 평가

## 결론

이 노트북은 비식별 합성 사례(치매·파킨슨병·뇌전증 재진 외래)에서 프롬프트 v0~v3의 정보 추출 성능을 비교한다. `frontend/`는 이 구조화 결과가 실제 문진 화면에 어떻게 쓰일지 보여주는 UI 프로토타입이며, NLP·LLM 실험은 이 노트북에서 수행한다. Flask(`legacy_flask/`)는 현재 범위 밖의 백엔드 후보 예시일 뿐이다.

평가 지표: Exact Match, Field Accuracy, Micro Precision/Recall/F1, 오류 유형 분석(누락 오류·과잉 추론·분류 오류·표현 오류·형식 오류)

실행 순서:

1. 환경 설정
2. API 키 입력
3. 사례와 프롬프트 확인
4. 사례 1개로 smoke test
5. 개발 세트에서 v0~v3 비교
6. 오류 분석과 프롬프트 수정
7. 선정된 프롬프트만 시험 세트에서 최종 평가, `results/`에 결과 저장


## 0. 데이터 사용 원칙

- 실제 환자의 이름, 생년월일, 등록번호, 연락처, 진료일을 입력하지 않는다.
- 외래 경험을 바탕으로 작성한 비식별 합성 문장만 사용한다.
- API 키는 코드에 직접 작성하거나 Git에 커밋하지 않는다.
- 사례 10개 결과는 소규모 파일럿 평가로 해석한다.


In [1]:
# 최초 한 번만 실행합니다. 설치 후 커널 재시작 안내가 나오면 재시작합니다.
%pip install -q openai python-dotenv pandas matplotlib

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
from datetime import datetime
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
pd.set_option("display.max_colwidth", 120)
print("환경 설정 완료")

환경 설정 완료


## 1. API 키 설정

로컬 프로젝트의 `.env`에 키가 있으면 자동으로 읽는다. 코랩처럼 `.env`가 없는 환경에서는 아래 셀 실행 후 입력창에 키를 붙여넣는다. 입력 내용은 화면에 표시되지 않는다.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY 입력: ")

client = OpenAI()
print("API 키 설정 완료")

## 2. 실험 설정

프롬프트 비교 중에는 모델을 고정한다. 모델 비교는 최종 프롬프트를 결정한 후 별도 실험으로 수행한다.

In [ ]:
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MAX_OUTPUT_TOKENS = 500
TEMPERATURE_NOTE = "Responses API 기본 설정 사용"

print({
    "model": MODEL,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "temperature": TEMPERATURE_NOTE,
})

## 3. 사례와 프롬프트 불러오기

노트북을 프로젝트 폴더에서 열면 `eval_cases.json`과 `prompts.py`를 자동으로 읽는다. 코랩에서는 다음 셀을 실행한 뒤 두 파일을 업로드한다.

In [ ]:
# 코랩에서만 실행합니다. 로컬 Jupyter에서는 실행하지 않아도 됩니다.
try:
    from google.colab import files
    print("eval_cases.json과 prompts.py를 선택하세요.")
    uploaded = files.upload()
except ImportError:
    print("로컬 Jupyter 환경입니다. 프로젝트 폴더의 파일을 사용합니다.")

In [ ]:
def find_project_dir():
    candidates = [Path.cwd(), Path.cwd() / "outputs" / "outpatient-ai-lab"]
    for candidate in candidates:
        if (candidate / "eval_cases.json").exists() and (candidate / "prompts.py").exists():
            return candidate
    raise FileNotFoundError(
        "eval_cases.json과 prompts.py를 찾지 못했습니다. "
        "노트북을 outpatient-ai-lab 폴더에서 실행하세요."
    )

PROJECT_DIR = find_project_dir()
RESULTS_DIR = PROJECT_DIR / "results"
RAW_DIR = RESULTS_DIR / "raw"
RESULTS_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

print("프로젝트 폴더:", PROJECT_DIR)
print("결과 저장 폴더:", RESULTS_DIR)


In [ ]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from prompts import ALLOWED_VALUES, PROMPT_VERSIONS

with (PROJECT_DIR / "eval_cases.json").open(encoding="utf-8") as file:
    EVAL_CASES = json.load(file)

print("사례 수:", len(EVAL_CASES))
print("프롬프트 버전:", list(PROMPT_VERSIONS))

In [ ]:
case_table = pd.DataFrame([
    {
        "case_id": case["case_id"],
        "disease_context": case.get("disease_context", ""),
        "split": case["split"],
        "input": case["input"],
        "gold": json.dumps(case["gold"], ensure_ascii=False),
    }
    for case in EVAL_CASES
])
case_table


### 사례 수정 방법

`eval_cases.json`에서 `input`과 `gold`를 수정한 후 위의 불러오기 셀부터 다시 실행한다. `input`에는 정돈되지 않은 구어체를 작성하고, `gold`에는 원문에서 직접 확인할 수 있는 값만 기록한다.

In [ ]:
for version, prompt in PROMPT_VERSIONS.items():
    print(f"\n===== {version} =====")
    print(prompt)

## 4. LLM 호출과 평가 함수

아래 셀은 실행 함수 정의다. 셀을 실행해도 아직 API 호출은 발생하지 않는다.

In [ ]:
from prompts import MULTI_LABEL_FIELDS, SINGLE_LABEL_FIELDS, FREE_TEXT_FIELDS

ALL_CLASSIFICATION_FIELDS = MULTI_LABEL_FIELDS + SINGLE_LABEL_FIELDS


def parse_json_object(raw_text):
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("JSON 객체를 찾지 못했습니다.")
    return json.loads(cleaned[start:end + 1])


def normalize_prediction(payload):
    normalized = {}

    for field in MULTI_LABEL_FIELDS:
        value = payload.get(field, [])
        if isinstance(value, str):
            value = [value]
        if not isinstance(value, list):
            value = []
        normalized[field] = [str(item).strip() for item in value if str(item).strip()]

    for field in ("symptom_change",):
        value = payload.get(field)
        normalized[field] = str(value).strip() if value not in (None, "") else None

    for field in ("patient_present", "guardian_only"):
        value = payload.get(field)
        normalized[field] = bool(value) if isinstance(value, bool) else None

    for field in FREE_TEXT_FIELDS:
        value = payload.get(field)
        normalized[field] = str(value).strip() if value not in (None, "") else None

    return normalized


In [ ]:
def run_llm(case, version):
    response = client.responses.create(
        model=MODEL,
        instructions=PROMPT_VERSIONS[version],
        input=(
            f"환자 또는 보호자 입력:\n{case['input']}\n\n"
            f"허용값:\n{json.dumps(ALLOWED_VALUES, ensure_ascii=False)}"
        ),
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    raw_output = response.output_text
    usage = {
        "input_tokens": getattr(response.usage, "input_tokens", None),
        "output_tokens": getattr(response.usage, "output_tokens", None),
        "total_tokens": getattr(response.usage, "total_tokens", None),
    }

    try:
        prediction = normalize_prediction(parse_json_object(raw_output))
        return prediction, raw_output, None, usage
    except Exception as error:
        return None, raw_output, str(error), usage


In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def empty_prediction():
    return {
        **{field: [] for field in MULTI_LABEL_FIELDS},
        "symptom_change": None,
        "patient_present": None,
        "guardian_only": None,
        **{field: None for field in FREE_TEXT_FIELDS},
    }


def field_is_correct(field, prediction, gold):
    if field in MULTI_LABEL_FIELDS:
        return set(prediction.get(field) or []) == set(gold.get(field) or [])
    return prediction.get(field) == gold.get(field)


def score_records(records):
    counts = {field: {"tp": 0, "fp": 0, "fn": 0} for field in MULTI_LABEL_FIELDS}
    single_correct = {field: 0 for field in SINGLE_LABEL_FIELDS}
    field_correct_total = 0
    field_total = 0
    exact_match_count = 0
    valid_count = 0

    for record in records:
        gold = record["gold"]
        prediction = record["prediction"]
        if prediction is not None:
            valid_count += 1
        else:
            prediction = empty_prediction()

        record_all_correct = True
        for field in MULTI_LABEL_FIELDS:
            gold_set = set(gold.get(field) or [])
            predicted_set = set(prediction.get(field) or [])
            counts[field]["tp"] += len(gold_set & predicted_set)
            counts[field]["fp"] += len(predicted_set - gold_set)
            counts[field]["fn"] += len(gold_set - predicted_set)

        for field in ALL_CLASSIFICATION_FIELDS:
            correct = field_is_correct(field, prediction, gold)
            field_total += 1
            field_correct_total += int(correct)
            if not correct:
                record_all_correct = False
            if field in SINGLE_LABEL_FIELDS:
                single_correct[field] += int(correct)

        exact_match_count += int(record_all_correct)

    total = {
        key: sum(field_counts[key] for field_counts in counts.values())
        for key in ("tp", "fp", "fn")
    }
    precision = safe_divide(total["tp"], total["tp"] + total["fp"])
    recall = safe_divide(total["tp"], total["tp"] + total["fn"])
    f1 = safe_divide(2 * precision * recall, precision + recall)

    metrics = {
        "case_count": len(records),
        "json_success_rate": safe_divide(valid_count, len(records)),
        "exact_match": safe_divide(exact_match_count, len(records)),
        "field_accuracy": safe_divide(field_correct_total, field_total),
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
        "omission_rate": safe_divide(total["fn"], total["tp"] + total["fn"]),
        "hallucination_rate": safe_divide(total["fp"], total["tp"] + total["fp"]),
        "input_tokens": sum((record.get("usage") or {}).get("input_tokens") or 0 for record in records),
        "output_tokens": sum((record.get("usage") or {}).get("output_tokens") or 0 for record in records),
    }
    for field in SINGLE_LABEL_FIELDS:
        metrics[f"{field}_accuracy"] = safe_divide(single_correct[field], len(records))

    return metrics


In [ ]:
def run_experiment(version, split="development"):
    selected_cases = [case for case in EVAL_CASES if case["split"] == split]
    records = []

    for index, case in enumerate(selected_cases, start=1):
        print(f"[{version}] {index}/{len(selected_cases)} {case['case_id']}")
        try:
            prediction, raw_output, error, usage = run_llm(case, version)
        except Exception as exception:
            prediction, raw_output, error, usage = None, "", str(exception), {}

        records.append({
            "case_id": case["case_id"],
            "disease_context": case.get("disease_context", ""),
            "split": split,
            "input": case["input"],
            "gold": case["gold"],
            "prediction": prediction,
            "raw_output": raw_output,
            "error": error,
            "usage": usage,
        })

    return {
        "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "model": MODEL,
        "prompt_version": version,
        "split": split,
        "metrics": score_records(records),
        "records": records,
    }


## 5. Smoke test — 사례 1개만 호출

아래 셀부터 실제 API 호출이 발생한다. 먼저 한 사례만 실행해 키, 모델명, JSON 변환을 확인한다.

In [ ]:
smoke_case = next(case for case in EVAL_CASES if case["split"] == "development")
smoke_prediction, smoke_raw, smoke_error, smoke_usage = run_llm(smoke_case, "v0")

print("입력:", smoke_case["input"])
print("정답:", json.dumps(smoke_case["gold"], ensure_ascii=False, indent=2))
print("예측:", json.dumps(smoke_prediction, ensure_ascii=False, indent=2))
print("오류:", smoke_error)
print("토큰:", smoke_usage)

## 6. 개발 세트에서 v0~v3 비교

아래 셀은 개발 사례 7개 × 프롬프트 4개로 총 28회 API를 호출한다. smoke test가 성공한 뒤 실행한다.

In [ ]:
development_results = {}

for version in PROMPT_VERSIONS:
    development_results[version] = run_experiment(version, split="development")

print("개발 세트 실험 완료")

In [ ]:
metrics_table = pd.DataFrame([
    {
        "version": version,
        **result["metrics"],
    }
    for version, result in development_results.items()
]).set_index("version")

metrics_table.style.format({
    "json_success_rate": "{:.1%}",
    "exact_match": "{:.1%}",
    "field_accuracy": "{:.1%}",
    "micro_precision": "{:.1%}",
    "micro_recall": "{:.1%}",
    "micro_f1": "{:.1%}",
    "symptom_change_accuracy": "{:.1%}",
    "patient_present_accuracy": "{:.1%}",
    "guardian_only_accuracy": "{:.1%}",
    "omission_rate": "{:.1%}",
    "hallucination_rate": "{:.1%}",
})


In [ ]:
chart_columns = [
    "exact_match",
    "field_accuracy",
    "micro_f1",
    "json_success_rate",
]

ax = metrics_table[chart_columns].plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
ax.set_title("Prompt version comparison — development set")
ax.set_ylabel("score")
ax.set_xlabel("prompt version")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. 사례별 오류 분석

전체 점수만 확인하면 프롬프트 수정 이유를 설명하기 어렵다. 정답과 예측이 다른 사례를 아래 5개 오류 유형으로 분류한다.

- **누락 오류**: 정답에 있는 값을 예측이 빠뜨림
- **과잉 추론**: 정답에 없는 값을 예측이 추가함(입력에 없는 내용을 지어냄)
- **분류 오류**: 단일 값 필드(symptom_change, patient_present, guardian_only)를 다른 값으로 잘못 분류함
- **표현 오류**: 자유 서술 필드(subjective_summary, requested_consultation, soap_summary)의 내용이나 어투가 부적절함 — 자동 채점이 어려워 수동 확인 필요
- **형식 오류**: LLM 출력이 JSON으로 파싱되지 않음


In [ ]:
def classify_error_types(record):
    prediction = record["prediction"]
    if prediction is None:
        return ["형식 오류"]

    gold = record["gold"]
    error_types = []

    for field in MULTI_LABEL_FIELDS:
        gold_set = set(gold.get(field) or [])
        predicted_set = set(prediction.get(field) or [])
        if gold_set - predicted_set:
            error_types.append("누락 오류")
        if predicted_set - gold_set:
            error_types.append("과잉 추론")

    for field in SINGLE_LABEL_FIELDS:
        gold_value = gold.get(field)
        predicted_value = prediction.get(field)
        if predicted_value == gold_value:
            continue
        if gold_value is not None and predicted_value is None:
            error_types.append("누락 오류")
        elif gold_value is None and predicted_value is not None:
            error_types.append("과잉 추론")
        else:
            error_types.append("분류 오류")

    return sorted(set(error_types))


def record_has_error(record):
    prediction = record["prediction"]
    if prediction is None:
        return True
    gold = record["gold"]
    for field in ALL_CLASSIFICATION_FIELDS:
        if not field_is_correct(field, prediction, gold):
            return True
    return False


def build_error_table(result):
    rows = []
    for record in result["records"]:
        if record_has_error(record):
            rows.append({
                "case_id": record["case_id"],
                "disease_context": record.get("disease_context", ""),
                "input": record["input"],
                "gold": json.dumps(record["gold"], ensure_ascii=False),
                "prediction": json.dumps(record["prediction"], ensure_ascii=False),
                "auto_error_types": ", ".join(classify_error_types(record)),
                "expression_check_needed": "표현 오류 수동 확인 필요",
                "manual_note": "직접 작성",
            })
    return pd.DataFrame(rows)


In [ ]:
# 확인할 버전을 변경할 수 있습니다.
ERROR_VERSION = "v0"
error_table = build_error_table(development_results[ERROR_VERSION])
error_table


### 수동 평가 기준

대표 사례에는 다음 항목을 0점 또는 1점으로 기록한다.

- 원문의 사실을 보존했는가
- 입력에 없는 사실을 생성하지 않았는가
- 불확실성을 확정적으로 바꾸지 않았는가
- 진료 전 확인 자료로 활용 가능한가


## 8. 개발 결과 저장

개발 세트의 지표표는 `results/metrics_summary.csv`에 저장한다(매 실행마다 최신 결과로 덮어씀). 상세 원본 응답은 `results/raw/`에 타임스탬프 파일로 남긴다. 저장 파일은 Git 커밋 전에 개인정보 포함 여부를 다시 확인한다.


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_json_path = RAW_DIR / f"{timestamp}_development_results.json"

with raw_json_path.open("w", encoding="utf-8") as file:
    json.dump(development_results, file, ensure_ascii=False, indent=2)

metrics_table.to_csv(RESULTS_DIR / "metrics_summary.csv", encoding="utf-8-sig")

print("상세 원본(raw):", raw_json_path)
print("지표 요약(포트폴리오용):", RESULTS_DIR / "metrics_summary.csv")


## 9. 최종 시험 세트 평가

개발 세트에서 프롬프트를 수정한 뒤 최종 버전 하나를 선택한다. 시험 세트에서는 여러 버전을 비교해 다시 선택하지 않는다.

In [ ]:
# 개발 결과를 확인한 후 최종 버전을 직접 지정합니다.
FINAL_VERSION = "v3"

test_result = run_experiment(FINAL_VERSION, split="test")
pd.DataFrame([{**test_result["metrics"], "version": FINAL_VERSION}]).set_index("version")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_test_path = RAW_DIR / f"{timestamp}_{FINAL_VERSION}_test.json"

with raw_test_path.open("w", encoding="utf-8") as file:
    json.dump(test_result, file, ensure_ascii=False, indent=2)

final_error_table = build_error_table(test_result)
final_error_table.to_csv(RESULTS_DIR / "error_analysis.csv", index=False, encoding="utf-8-sig")

print("상세 원본(raw):", raw_test_path)
print("최종 오류 분석(포트폴리오용):", RESULTS_DIR / "error_analysis.csv")


## 10. 결과 해석 작성란

실행 후 아래 내용을 직접 작성한다.

### 기준선
- v0의 주요 점수:
- 반복된 오류:

### 프롬프트 변경
- 추가한 지시:
- 변경 근거:

### 결과 변화
- 개선된 지표:
- 악화된 지표:
- 대표 성공 사례:
- 대표 실패 사례:

### 한계
- 사례가 10개인 소규모 파일럿 평가다.
- 합성 데이터이므로 실제 외래 표현 분포를 대표하지 않는다.
- 최종 판단을 위한 임상 성능 평가가 아니라 정보 추출 프로토타입 평가다.


## 11. 포트폴리오 작성 틀

> 외래 간호 경험에서 확인한 정보 누락, 서류 요청 지연, 보호자 단독 내원, 상담 내용 누락 문제를 바탕으로 신경과(치매·파킨슨병·뇌전증) 재진 환자 대상 AI 사전 문진을 설계하였다. 비식별 합성 사례와 gold 데이터를 구축하고, 동일한 모델에서 프롬프트 v0~v3의 Exact Match, Field Accuracy, Micro F1을 비교하였다. 대표 오류를 5개 유형(누락 오류·과잉 추론·분류 오류·표현 오류·형식 오류)으로 분석해 [수정한 지시]를 추가했으며, 개발 세트의 [지표]가 [수치]에서 [수치]로 변화하였다. 최종 프롬프트는 보지 않은 시험 세트에서 평가하고, `frontend/` 문진 UI 프로토타입으로 적용 흐름을 시연하였다.

대괄호 안에는 실제 실행 결과만 작성한다.


# 결론

이 노트북의 완료 기준은 v0~v3 점수표, 대표 오류 분석, 최종 버전 선정 근거, 시험 세트 결과가 모두 남는 것이다. LangSmith는 이 로컬 평가 흐름이 정상 작동한 뒤 같은 데이터셋과 평가 함수를 실험 관리 화면에 연결한다.